# 0. Import

In [50]:
!pip install wandb -q

In [51]:
import json
import wandb
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
import json
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn

In [52]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

# Fetch the secret token safely
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

# Log into WandB
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

# 1. Making the Modality

In [54]:
def load_skeleton(trial_path):

    prediction_dir = Path(trial_path) / "predictions"

    json_files = sorted(prediction_dir.glob("*.json"))

    frames = []

    for json_file in json_files:

        with open(json_file, "r") as f:
            data = json.load(f)

        # One person per frame based on our inspection
        person = data[0]

        keypoints = np.asarray(
            person["keypoints"],
            dtype=np.float32
        )

        frames.append(keypoints)

    if len(frames) == 0:
        return None

    return np.stack(frames)

## 1. Building the training index

In [55]:
def temporal_resample(x, target_frames=64):

    T = x.shape[0]

    if T == target_frames:
        return x.astype(np.float32)

    old_indices = np.linspace(
        0,
        T - 1,
        T
    )

    new_indices = np.linspace(
        0,
        T - 1,
        target_frames
    )

    output = np.empty(
        (target_frames, x.shape[1], x.shape[2]),
        dtype=np.float32
    )

    for joint in range(x.shape[1]):
        for coord in range(x.shape[2]):

            output[:, joint, coord] = np.interp(
                new_indices,
                old_indices,
                x[:, joint, coord]
            )

    return output

In [56]:
def normalize_skeleton(x):
    """
    x: (T, 17, 3)

    Makes skeleton coordinates root-relative.
    Joint 0 is treated as the root.
    """

    x = x.copy()

    # Root-relative coordinates
    root = x[:, 0:1, :]
    x = x - root

    return x.astype(np.float32)

def temporal_resample(x, target_frames=64):
    """
    Resample the complete sequence to exactly target_frames.
    """

    T = x.shape[0]

    if T == target_frames:
        return x.astype(np.float32)

    if T == 1:
        return np.repeat(
            x,
            target_frames,
            axis=0
        ).astype(np.float32)

    old_indices = np.linspace(
        0,
        T - 1,
        T
    )

    new_indices = np.linspace(
        0,
        T - 1,
        target_frames
    )

    output = np.empty(
        (target_frames, x.shape[1], x.shape[2]),
        dtype=np.float32
    )

    for joint in range(x.shape[1]):

        for coord in range(x.shape[2]):

            output[:, joint, coord] = np.interp(
                new_indices,
                old_indices,
                x[:, joint, coord]
            )

    return output.astype(np.float32)

def get_bone_vectors(self, x):
    """
    x: (T, 17, 3)

    Returns:
        bone_vectors: (T, 17, 3)
    """

    # COCO-17 skeleton hierarchy
    #
    # 0  nose
    # 1  left eye
    # 2  right eye
    # 3  left ear
    # 4  right ear
    # 5  left shoulder
    # 6  right shoulder
    # 7  left elbow
    # 8  right elbow
    # 9  left wrist
    # 10 right wrist
    # 11 left hip
    # 12 right hip
    # 13 left knee
    # 14 right knee
    # 15 left ankle
    # 16 right ankle

    parents = [
        -1,  # nose
         0,  # left eye
         0,  # right eye
         1,  # left ear
         2,  # right ear
        11,  # left shoulder -> left hip
        12,  # right shoulder -> right hip
         5,  # left elbow
         6,  # right elbow
         7,  # left wrist
         8,  # right wrist
        -1,  # left hip
        -1,  # right hip
        11,  # left knee
        12,  # right knee
        13,  # left ankle
        14   # right ankle
    ]

    bone_vectors = np.zeros_like(x)

    for joint, parent in enumerate(parents):

        if parent != -1:

            bone_vectors[:, joint, :] = (
                x[:, joint, :] -
                x[:, parent, :]
            )

    return bone_vectors

## 3. Dataset Class

In [57]:
class SkeletonDataset(Dataset):
    def __init__(self, df, sequence_length=64):
        self.df = df.reset_index(drop=True)
        self.sequence_length = sequence_length

    def load_skeleton(self, path):
        prediction_dir = Path(path) / "predictions"
        json_files = sorted(prediction_dir.glob("*.json"))

        keypoint_frames = []
        confidence_frames = []

        for json_file in json_files:
            with open(json_file, "r") as f:
                data = json.load(f)

            if len(data) == 0:
                continue

            person = data[0]

            # -------------------------
            # Keypoints
            # -------------------------
            keypoints = np.asarray(
                person["keypoints"],
                dtype=np.float32
            )

            # Expected: (17, 3)
            if keypoints.shape != (17, 3):
                continue

            # -------------------------
            # Confidence scores
            # -------------------------
            scores = np.asarray(
                person["keypoint_scores"],
                dtype=np.float32
            ).reshape(-1)

            # Expected: 17 scores
            if scores.shape[0] != 17:
                continue

            scores = np.clip(scores, 0.0, 1.0)

            keypoint_frames.append(keypoints)
            confidence_frames.append(scores)

        if len(keypoint_frames) == 0:
            return None, None

        keypoints = np.stack(keypoint_frames)       # (T,17,3)
        scores = np.stack(confidence_frames)        # (T,17)

        return keypoints, scores

    # --------------------------------------------------
    # Pelvis-centered + scale normalization
    # --------------------------------------------------
    def normalize_skeleton(self, x):
        x = x.copy()

        # COCO-17
        # left hip  = 11
        # right hip = 12
        # left shoulder  = 5
        # right shoulder = 6

        pelvis = (
            x[:, 11:12, :] +
            x[:, 12:13, :]
        ) / 2.0

        x = x - pelvis

        shoulder_center = (
            x[:, 5:6, :] +
            x[:, 6:7, :]
        ) / 2.0

        scale = np.linalg.norm(
            shoulder_center,
            axis=2,
            keepdims=True
        )

        scale = np.maximum(scale, 1e-6)

        x = x / scale

        return x

    # --------------------------------------------------
    # Bone vectors
    # --------------------------------------------------
    def get_bone_vectors(self, x):

        parents = [
            -1, 0, 0, 1, 2,
            11, 12,
            5, 6,
            7, 8,
            -1, -1,
            11, 12,
            13, 14
        ]

        bones = np.zeros_like(x)

        for j, p in enumerate(parents):
            if p >= 0:
                bones[:, j, :] = x[:, j, :] - x[:, p, :]

        return bones

    # --------------------------------------------------
    # Temporal resampling
    # --------------------------------------------------
    def temporal_resample(self, x):

        T = x.shape[0]

        if T == self.sequence_length:
            return x.astype(np.float32)

        if T == 1:
            return np.repeat(
                x,
                self.sequence_length,
                axis=0
            ).astype(np.float32)

        old_indices = np.linspace(0, T - 1, T)
        new_indices = np.linspace(
            0,
            T - 1,
            self.sequence_length
        )

        output = np.empty(
            (
                self.sequence_length,
                x.shape[1],
                x.shape[2]
            ),
            dtype=np.float32
        )

        for joint in range(x.shape[1]):
            for coord in range(x.shape[2]):
                output[:, joint, coord] = np.interp(
                    new_indices,
                    old_indices,
                    x[:, joint, coord]
                )

        return output

    # --------------------------------------------------
    # Dataset
    # --------------------------------------------------
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        skeleton, confidence = self.load_skeleton(
            row["skeleton"]
        )

        # Missing skeleton
        if skeleton is None:

            skeleton = np.zeros(
                (self.sequence_length, 17, 3),
                dtype=np.float32
            )

            confidence = np.zeros(
                (self.sequence_length, 17),
                dtype=np.float32
            )

        # -------------------------
        # Normalize coordinates
        # -------------------------
        skeleton = self.normalize_skeleton(skeleton)

        # -------------------------
        # Velocity
        # -------------------------
        velocity = np.diff(
            skeleton,
            axis=0,
            prepend=skeleton[0:1]
        )

        # -------------------------
        # Acceleration
        # -------------------------
        acceleration = np.diff(
            velocity,
            axis=0,
            prepend=velocity[0:1]
        )

        # -------------------------
        # Bone vectors
        # -------------------------
        bones = self.get_bone_vectors(skeleton)

        # -------------------------
        # Add confidence
        #
        # XYZ        = 3
        # velocity   = 3
        # acceleration = 3
        # bones      = 3
        # confidence = 1
        #
        # TOTAL = 13 features/joint
        # -------------------------


        features = np.concatenate(
            [
                skeleton,
                velocity,
                acceleration,
                bones,
            ],
            axis=2
        )

        # (T, 17, 13)

        features = self.temporal_resample(features)

        # Final shape:
        # (64, 17, 13)

        X = torch.tensor(
            features,
            dtype=torch.float32
        )

        y = torch.tensor(
            row["label"],
            dtype=torch.long
        )

        return X, y

In [58]:
from pathlib import Path
import pandas as pd

TRAIN_DATA = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "small-model-track/Training/Training/data"
)

MODALITIES = [
    "Skeleton",
    "Depth_Color",
    "IR",
    "Thermal",
    "IMU",
    "Radar",
]

# ---------------------------------------------------------
# Build trial index from Skeleton
# ---------------------------------------------------------

records = []

skeleton_root = TRAIN_DATA / "Skeleton"

for action_dir in sorted(skeleton_root.iterdir()):

    if not action_dir.is_dir():
        continue

    action_name = action_dir.name
    label = int(action_name.split("_")[0])

    for user_dir in sorted(action_dir.iterdir()):

        if not user_dir.is_dir():
            continue

        user = user_dir.name

        for trial_dir in sorted(user_dir.iterdir()):

            if not trial_dir.is_dir():
                continue

            records.append({
                "action": action_name,
                "label": label,
                "user": user,
                "trial": trial_dir.name,
                "path": str(trial_dir),
            })


train_df = pd.DataFrame(records)

print("Training trials:", len(train_df))
print("Classes:", train_df["label"].nunique())
print("Users:", train_df["user"].nunique())

train_df.head()

Training trials: 2931
Classes: 40
Users: 18


,action,label,user,trial,path
0,0_Wash_face,0,user16,1-1-1,/kaggle/input/datasets/samasiayushman/small-mo...
1,0_Wash_face,0,user16,1-1-2,/kaggle/input/datasets/samasiayushman/small-mo...
2,0_Wash_face,0,user16,1-1-3,/kaggle/input/datasets/samasiayushman/small-mo...
3,0_Wash_face,0,user18,7-1-1,/kaggle/input/datasets/samasiayushman/small-mo...
4,0_Wash_face,0,user18,7-1-2,/kaggle/input/datasets/samasiayushman/small-mo...


In [59]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_df,
        train_df["label"],
        groups=train_df["user"]
    )
)

train_split = train_df.iloc[train_idx].reset_index(drop=True)
val_split = train_df.iloc[val_idx].reset_index(drop=True)

print("Train:", len(train_split))
print("Validation:", len(val_split))

print("Train users:")
print(sorted(train_split["user"].unique()))

print("\nValidation users:")
print(sorted(val_split["user"].unique()))

Train: 2238
Validation: 693
Train users:
['user17', 'user18', 'user19', 'user20', 'user21', 'user23', 'user24', 'user3', 'user4', 'user5', 'user6', 'user7', 'user8', 'user9']

Validation users:
['user1', 'user16', 'user2', 'user22']


In [60]:
train_dataset = SkeletonDataset(
    train_split.assign(skeleton=train_split["path"]),
    sequence_length=64
)

val_dataset = SkeletonDataset(
    val_split.assign(skeleton=val_split["path"]),
    sequence_length=64
)

print("Train dataset:", len(train_dataset))
print("Val dataset:", len(val_dataset))

Train dataset: 2238
Val dataset: 693


In [61]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

In [62]:
X, y = next(iter(train_loader))

print("Train batch X:", X.shape)
print("Train batch y:", y.shape)

X, y = next(iter(val_loader))

print("Val batch X:", X.shape)
print("Val batch y:", y.shape)

Train batch X: torch.Size([32, 64, 17, 12])
Train batch y: torch.Size([32])
Val batch X: torch.Size([32, 64, 17, 12])
Val batch y: torch.Size([32])


In [63]:
dataset =  SkeletonDataset(
    train_split.assign(skeleton=train_split["path"]),
    sequence_length=64
)


X, y = dataset[0]

print("X shape:", X.shape)
print("y:", y)
print("dtype:", X.dtype)

print("Min:", X.min().item())
print("Max:", X.max().item())
print("Mean:", X.mean().item())
print("Std:", X.std().item())

X shape: torch.Size([64, 17, 12])
y: tensor(0)
dtype: torch.float32
Min: -1.2548960447311401
Max: 1.66280198097229
Mean: -0.0242230873554945
Std: 0.29696953296661377


---

In [64]:
print("Root joint mean:", X[:, 0, :].abs().mean().item())

print( "Root joint max:", X[:, 0, :].abs().max().item())

Root joint mean: 0.10289439558982849
Root joint max: 0.6045839190483093


# A. IMU

In [65]:
IMU_ROOT = "/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/IMU"

print(os.path.exists(IMU_ROOT))

True


In [66]:
SENSOR_ORDER = [
    "WTLA",   # Left Arm
    "WTRA",   # Right Arm
    "WTC",    # Chest
    "WTLL",   # Left Leg
    "WTRL"    # Right Leg
]
IMU_FEATURES = [
    "加速度X(g)",
    "加速度Y(g)",
    "加速度Z(g)",
    "角速度X(°/s)",
    "角速度Y(°/s)",
    "角速度Z(°/s)"
]
def load_imu_trial(trial_dir):

    sensor_data = {}

    for fname in [
        "up(LA+RA+C).csv",
        "down(LL+RL).csv"
    ]:

        path = os.path.join(trial_dir, fname)

        if not os.path.exists(path):
            continue

        df = pd.read_csv(path)

        df["时间"] = pd.to_datetime(df["时间"])

        for device in df["设备名称"].unique():

            # Identify sensor from device name
            if device.startswith("WTLA"):
                sensor = "WTLA"
            elif device.startswith("WTRA"):
                sensor = "WTRA"
            elif device.startswith("WTC"):
                sensor = "WTC"
            elif device.startswith("WTLL"):
                sensor = "WTLL"
            elif device.startswith("WTRL"):
                sensor = "WTRL"
            else:
                continue

            sensor_df = df[df["设备名称"] == device].copy()

            sensor_df = sensor_df.sort_values("时间")

            sensor_data[sensor] = sensor_df[
                ["时间"] + IMU_FEATURES
            ].reset_index(drop=True)

    return sensor_data

In [67]:
test_trial = os.path.join(
    IMU_ROOT,
    "0_Wash_face",
    "user16",
    "1-1-1"
)

sensor_data = load_imu_trial(test_trial)

for sensor in SENSOR_ORDER:
    df = sensor_data[sensor]

    print(
        sensor,
        "shape =", df.shape,
        "start =", df["时间"].min(),
        "end =", df["时间"].max()
    )

WTLA shape = (45, 17) start = 2025-06-10 10:43:49.117000 end = 2025-06-10 10:43:53.607000
WTRA shape = (46, 17) start = 2025-06-10 10:43:49.143000 end = 2025-06-10 10:43:53.586000
WTC shape = (46, 17) start = 2025-06-10 10:43:49.214000 end = 2025-06-10 10:43:53.550000
WTLL shape = (44, 17) start = 2025-06-10 10:43:49.169000 end = 2025-06-10 10:43:53.519000
WTRL shape = (46, 17) start = 2025-06-10 10:43:49.131000 end = 2025-06-10 10:43:53.542000


In [68]:
SEQUENCE_LENGTH = 64

def synchronize_imu(sensor_data, sequence_length=64):
    """
    Synchronize 5 asynchronous IMU streams onto a common timeline.

    Output:
        shape = (sequence_length, 30)
    """

    # Find the common overlapping time interval
    start_time = max(
        sensor_data[sensor]["时间"].min()
        for sensor in SENSOR_ORDER
    )

    end_time = min(
        sensor_data[sensor]["时间"].max()
        for sensor in SENSOR_ORDER
    )

    # Convert timestamps to seconds relative to start_time
    common_times = np.linspace(
        0,
        (end_time - start_time).total_seconds(),
        sequence_length
    )

    all_features = []

    for sensor in SENSOR_ORDER:

        df = sensor_data[sensor].copy()

        # Time in seconds relative to common start
        time_seconds = (
            df["时间"] - start_time
        ).dt.total_seconds().to_numpy()

        sensor_features = []

        for feature in IMU_FEATURES:

            values = df[feature].astype(float).to_numpy()

            # Interpolate onto common timeline
            interpolated = np.interp(
                common_times,
                time_seconds,
                values
            )

            sensor_features.append(interpolated)

        # (64, 6)
        sensor_features = np.stack(
            sensor_features,
            axis=1
        )

        all_features.append(sensor_features)

    # 5 × (64, 6)
    # → (64, 30)
    synchronized = np.concatenate(
        all_features,
        axis=1
    )

    return synchronized.astype(np.float32)

In [69]:
imu_sample = synchronize_imu(sensor_data)

print("Shape:", imu_sample.shape)
print("dtype:", imu_sample.dtype)
print("min:", imu_sample.min())
print("max:", imu_sample.max())

feature_names = []

for sensor in SENSOR_ORDER:
    for feature in ["ax", "ay", "az", "gx", "gy", "gz"]:
        feature_names.append(f"{sensor}_{feature}")

print(len(feature_names))
print(feature_names)

Shape: (64, 80)
dtype: float32
min: -177.16536
max: 172.17143
30
['WTLA_ax', 'WTLA_ay', 'WTLA_az', 'WTLA_gx', 'WTLA_gy', 'WTLA_gz', 'WTRA_ax', 'WTRA_ay', 'WTRA_az', 'WTRA_gx', 'WTRA_gy', 'WTRA_gz', 'WTC_ax', 'WTC_ay', 'WTC_az', 'WTC_gx', 'WTC_gy', 'WTC_gz', 'WTLL_ax', 'WTLL_ay', 'WTLL_az', 'WTLL_gx', 'WTLL_gy', 'WTLL_gz', 'WTRL_ax', 'WTRL_ay', 'WTRL_az', 'WTRL_gx', 'WTRL_gy', 'WTRL_gz']


In [70]:
def collect_imu_trials(root_dir):
    trials = []

    for action_name in sorted(os.listdir(root_dir)):

        action_dir = os.path.join(root_dir, action_name)

        if not os.path.isdir(action_dir):
            continue

        for user_name in sorted(os.listdir(action_dir)):

            user_dir = os.path.join(action_dir, user_name)

            if not os.path.isdir(user_dir):
                continue

            for trial_name in sorted(os.listdir(user_dir)):

                trial_dir = os.path.join(user_dir, trial_name)

                if not os.path.isdir(trial_dir):
                    continue

                up_file = os.path.join(
                    trial_dir,
                    "up(LA+RA+C).csv"
                )

                down_file = os.path.join(
                    trial_dir,
                    "down(LL+RL).csv"
                )

                if os.path.exists(up_file) and os.path.exists(down_file):

                    trials.append({
                        "action": action_name,
                        "user": user_name,
                        "trial_id": f"{user_name}_{trial_name}",
                        "trial_dir": trial_dir
                    })

    return pd.DataFrame(trials)

imu_df = collect_imu_trials(IMU_ROOT)

print("Total trials:", len(imu_df))
print(imu_df.head())
print("\nActions:", imu_df["action"].nunique())
print("Users:", imu_df["user"].nunique())

Total trials: 2842
        action    user      trial_id  \
0  0_Wash_face  user16  user16_1-1-1   
1  0_Wash_face  user16  user16_1-1-2   
2  0_Wash_face  user16  user16_1-1-3   
3  0_Wash_face  user18  user18_7-1-1   
4  0_Wash_face  user18  user18_7-1-2   

                                           trial_dir  
0  /kaggle/input/datasets/samasiayushman/small-mo...  
1  /kaggle/input/datasets/samasiayushman/small-mo...  
2  /kaggle/input/datasets/samasiayushman/small-mo...  
3  /kaggle/input/datasets/samasiayushman/small-mo...  
4  /kaggle/input/datasets/samasiayushman/small-mo...  

Actions: 40
Users: 18


In [71]:
print(
    imu_df["action"]
    .value_counts()
    .sort_index()
)

action
0_Wash_face                            43
10_Stir_drinks                        113
11_Peel_fruits                        102
12_Sweep_the_floor                     62
13_Mop_the_floor                       53
14_Wipe_bowls                          35
15_Wipe_windows_and_tables             48
16_Fold_clothes                        24
17_Tap_the_keyboard                    84
18_Write                               34
19_Make_a_phone_call                   39
1_Brush_teeth                          48
20_Check_the_time                      94
21_Read_documents                      70
22_Turn_pages                          56
23_Listen_to_music_with_headphones     74
24_Use_a_mobile_phone                  46
25_Watch_TV                            12
26_Play_games                          40
27_Take_a_selfie                       41
28_Jog_in_place                        35
29_Do_squats                           74
2_Comb_hair                            54
30_Do_jumping_jacks        

In [72]:
print(sorted(imu_df["user"].unique()))
print("\nTrials per user:")
print(imu_df["user"].value_counts().sort_index())

['user1', 'user16', 'user17', 'user18', 'user19', 'user2', 'user20', 'user21', 'user22', 'user23', 'user24', 'user3', 'user4', 'user5', 'user6', 'user7', 'user8', 'user9']

Trials per user:
user
user1     139
user16    186
user17    154
user18    178
user19    183
user2     146
user20    159
user21    130
user22    185
user23    131
user24    159
user3     147
user4     124
user5      90
user6     201
user7     184
user8     165
user9     181
Name: count, dtype: int64


In [73]:
# ============================================================
# IMU I1 — SUBJECT-HELD-OUT SPLIT
# ============================================================

VAL_USERS = {"user8", "user9", "user23", "user24"}

imu_train_df = imu_df[~imu_df["user"].isin(VAL_USERS)].reset_index(drop=True)
imu_val_df   = imu_df[ imu_df["user"].isin(VAL_USERS)].reset_index(drop=True)

print("Training trials:", len(imu_train_df))
print("Validation trials:", len(imu_val_df))

print("\nTraining users:")
print(sorted(imu_train_df["user"].unique()))

print("\nValidation users:")
print(sorted(imu_val_df["user"].unique()))

print("\nValidation class distribution:")
print(imu_val_df["action"].value_counts().sort_index())

Training trials: 2206
Validation trials: 636

Training users:
['user1', 'user16', 'user17', 'user18', 'user19', 'user2', 'user20', 'user21', 'user22', 'user3', 'user4', 'user5', 'user6', 'user7']

Validation users:
['user23', 'user24', 'user8', 'user9']

Validation class distribution:
action
0_Wash_face                           15
10_Stir_drinks                        19
11_Peel_fruits                        15
12_Sweep_the_floor                    12
13_Mop_the_floor                       8
14_Wipe_bowls                          3
15_Wipe_windows_and_tables            12
16_Fold_clothes                        9
17_Tap_the_keyboard                   22
18_Write                              11
19_Make_a_phone_call                  11
1_Brush_teeth                         16
20_Check_the_time                     16
21_Read_documents                     21
22_Turn_pages                         12
23_Listen_to_music_with_headphones    21
24_Use_a_mobile_phone                  9
25_Watch_T

In [74]:
# ============================================================
# CLASS MAPPING
# ============================================================

ACTION_NAMES = sorted(imu_df["action"].unique())

action_to_idx = {
    action: idx
    for idx, action in enumerate(ACTION_NAMES)
}

idx_to_action = {
    idx: action
    for action, idx in action_to_idx.items()
}

print("Number of classes:", len(ACTION_NAMES))
print("\nFirst 10 classes:")
for i in range(10):
    print(i, ACTION_NAMES[i])

Number of classes: 40

First 10 classes:
0 0_Wash_face
1 10_Stir_drinks
2 11_Peel_fruits
3 12_Sweep_the_floor
4 13_Mop_the_floor
5 14_Wipe_bowls
6 15_Wipe_windows_and_tables
7 16_Fold_clothes
8 17_Tap_the_keyboard
9 18_Write


In [75]:
class IMUDataset(torch.utils.data.Dataset):

    def __init__(self, df, action_to_idx):
        self.df = df.reset_index(drop=True)
        self.action_to_idx = action_to_idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        sensor_data = load_imu_trial(row["trial_dir"])

        # Synchronize all 5 sensors
        features = synchronize_imu(
            sensor_data,
            sequence_length=SEQUENCE_LENGTH
        )

        X = torch.tensor(features, dtype=torch.float32)

        y = torch.tensor(
            self.action_to_idx[row["action"]],
            dtype=torch.long
        )

        return X, y

In [76]:
imu_train_dataset = IMUDataset(
    imu_train_df,
    action_to_idx
)

X, y = imu_train_dataset[0]

print("X shape:", X.shape)
print("X dtype:", X.dtype)
print("y:", y)
print("y dtype:", y.dtype)

X shape: torch.Size([64, 80])
X dtype: torch.float32
y: tensor(0)
y dtype: torch.int64


In [77]:
def find_incomplete_imu_trials(df):

    incomplete = []

    for i, row in df.iterrows():

        try:
            sensor_data = load_imu_trial(row["trial_dir"])

            missing = [
                sensor
                for sensor in SENSOR_ORDER
                if sensor not in sensor_data
            ]

            if missing:
                incomplete.append({
                    "index": i,
                    "action": row["action"],
                    "user": row["user"],
                    "trial_id": row["trial_id"],
                    "missing": missing
                })

        except Exception as e:
            incomplete.append({
                "index": i,
                "action": row["action"],
                "user": row["user"],
                "trial_id": row["trial_id"],
                "missing": [f"ERROR: {e}"]
            })

    return pd.DataFrame(incomplete)


incomplete_imu = find_incomplete_imu_trials(imu_df)

print("Incomplete trials:", len(incomplete_imu))

if len(incomplete_imu) > 0:
    print(incomplete_imu.head(20).to_string(index=False))

Incomplete trials: 87
 index                     action   user     trial_id                       missing
     6                0_Wash_face user20 user20_4-2-1                  [WTLL, WTRL]
    48             10_Stir_drinks  user1  user1_4-1-1                        [WTLL]
    91             10_Stir_drinks user20 user20_5-1-3                  [WTLL, WTRL]
   158             11_Peel_fruits  user1  user1_4-1-1                        [WTLL]
   272         12_Sweep_the_floor user19 user19_3-3-1                  [WTLL, WTRL]
   293         12_Sweep_the_floor  user3  user3_6-2-1 [WTLA, WTRA, WTC, WTLL, WTRL]
   294         12_Sweep_the_floor  user3  user3_6-2-2 [WTLA, WTRA, WTC, WTLL, WTRL]
   295         12_Sweep_the_floor  user3  user3_6-2-3 [WTLA, WTRA, WTC, WTLL, WTRL]
   332           13_Mop_the_floor user19 user19_3-3-1                  [WTLL, WTRL]
   375              14_Wipe_bowls  user1  user1_4-1-1                  [WTLL, WTRL]
   435 15_Wipe_windows_and_tables  user5  user5_5-1-2 

In [78]:
complete_imu_df = imu_df.copy()

incomplete_indices = set(incomplete_imu["index"].tolist())

complete_imu_df = complete_imu_df[
    ~complete_imu_df.index.isin(incomplete_indices)
].reset_index(drop=True)

print("Original trials:", len(imu_df))
print("Incomplete trials:", len(incomplete_imu))
print("Complete trials:", len(complete_imu_df))

Original trials: 2842
Incomplete trials: 87
Complete trials: 2755


In [79]:
VAL_USERS = {"user8", "user9", "user23", "user24"}

imu_train_df = complete_imu_df[
    ~complete_imu_df["user"].isin(VAL_USERS)
].reset_index(drop=True)

imu_val_df = complete_imu_df[
    complete_imu_df["user"].isin(VAL_USERS)
].reset_index(drop=True)

print("Train:", len(imu_train_df))
print("Val:", len(imu_val_df))

Train: 2124
Val: 631


In [80]:
def calculate_imu_normalization(dataset, max_samples=1000, seed=42):
    rng = np.random.default_rng(seed)

    n = min(len(dataset), max_samples)
    indices = rng.choice(len(dataset), size=n, replace=False)

    all_features = []

    for i in indices:
        X, _ = dataset[i]
        all_features.append(X.numpy())

    all_features = np.concatenate(all_features, axis=0)

    mean = all_features.mean(axis=0)
    std = all_features.std(axis=0)

    std = np.maximum(std, 1e-6)

    return mean.astype(np.float32), std.astype(np.float32)

In [81]:
train_dataset_raw = IMUDataset(
    imu_train_df,
    action_to_idx
)

val_dataset_raw = IMUDataset(
    imu_val_df,
    action_to_idx
)

imu_mean, imu_std = calculate_imu_normalization(
    train_dataset_raw,
    max_samples=1000
)

print("Mean shape:", imu_mean.shape)
print("Std shape:", imu_std.shape)

Mean shape: (80,)
Std shape: (80,)


In [82]:

class NormalizedIMUDataset(IMUDataset):

    def __init__(self, df, action_to_idx, mean, std):
        super().__init__(df, action_to_idx)

        self.mean = torch.tensor(mean, dtype=torch.float32)
        self.std = torch.tensor(std, dtype=torch.float32)

    def __getitem__(self, idx):

        X, y = super().__getitem__(idx)

        X = (X - self.mean) / self.std

        return X, y

train_dataset = NormalizedIMUDataset(
    imu_train_df,
    action_to_idx,
    imu_mean,
    imu_std
)

val_dataset = NormalizedIMUDataset(
    imu_val_df,
    action_to_idx,
    imu_mean,
    imu_std
)

In [83]:
train_dataset = NormalizedIMUDataset(
    imu_train_df,
    action_to_idx,
    imu_mean,
    imu_std
)

val_dataset = NormalizedIMUDataset(
    imu_val_df,
    action_to_idx,
    imu_mean,
    imu_std
)

In [84]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [85]:
class I2IMUModel(nn.Module):
    def __init__(
        self,
        input_size=30,
        projection_size=128,
        hidden_size=128,
        num_layers=2,
        num_heads=4,
        num_classes=40,
        dropout=0.3
    ):
        super().__init__()

        # Input projection
        self.input_projection = nn.Sequential(
            nn.Linear(input_size, projection_size),
            nn.LayerNorm(projection_size),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # Temporal encoder
        self.lstm = nn.LSTM(
            input_size=projection_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        feature_size = hidden_size * 2

        # Temporal self-attention
        self.temporal_attention = nn.MultiheadAttention(
            embed_dim=feature_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(feature_size)

        # Learn which frames matter
        self.frame_attention = nn.Sequential(
            nn.Linear(feature_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        # x: (B, 64, 30)

        x = self.input_projection(x)

        # (B, 64, 128)
        x, _ = self.lstm(x)

        # Self-attention
        attended, _ = self.temporal_attention(
            x, x, x
        )

        # Residual connection + normalization
        x = self.norm(x + attended)

        # Learn importance of each frame
        scores = self.frame_attention(x)

        # (B, 64, 1)
        weights = torch.softmax(scores, dim=1)

        # Weighted temporal pooling
        x = torch.sum(x * weights, dim=1)

        # (B, 256) → classes
        logits = self.classifier(x)

        return logits

In [86]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = I2IMUModel().to(device)

print("Device:", device)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("Parameters:", total_params)
print("Approx FP32 size:", total_params * 4 / 1024**2, "MB")

Device: cpu
Parameters: 1004841
Approx FP32 size: 3.8331642150878906 MB


In [87]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20
)


def train_one_epoch(model, loader):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for X, y in loader:

        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(X)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * X.size(0)

        predictions = logits.argmax(dim=1)

        correct += (predictions == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    for X, y in loader:

        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(X)

        loss = criterion(logits, y)

        total_loss += loss.item() * X.size(0)

        predictions = logits.argmax(dim=1)

        correct += (predictions == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total

In [88]:
EPOCHS = 20

best_val_acc = 0.0
best_state = None
history = []

wandbinit(
    "Better Features + IMU I2 Better Temporal",
    "Multihead Attention BiLSTM"
)

for epoch in range(1, EPOCHS + 1):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader
    )

    val_loss, val_acc = evaluate(
        model,
        val_loader
    )

    scheduler.step()

    # -------------------------
    # W&B logging
    # -------------------------
    wandb.log({
        "epoch": epoch,
        "train/loss": train_loss,
        "train/accuracy": train_acc,
        "val/loss": val_loss,
        "val/accuracy": val_acc,
        "learning_rate": optimizer.param_groups[0]["lr"]
    })

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        best_state = {
            k: v.cpu().clone()
            for k, v in model.state_dict().items()
        }

        # Log best result
        wandb.log({
            "best/val_accuracy": best_val_acc,
            "best/epoch": epoch
        })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

print(
    f"\nBest I2 validation accuracy: "
    f"{best_val_acc:.4f}"
)

best/epoch,▁▂▃▅▇█
best/val_accuracy,▁▄▇▇▇█
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
learning_rate,███▇▇▇▆▆▅▅▄▃▃▂▂▂▁▁▁▁
train/accuracy,▁▂▃▃▄▅▅▆▆▇▇▇▇███████
train/loss,█▇▆▅▄▄▃▃▃▂▂▂▁▁▁▁▁▁▁▁
val/accuracy,▁▄▂▃▇▄▇▇▇▆▆▇█▇▇▇████
val/loss,▁▁▁▂▂▃▃▄▄▆▆▆▇▇▇█████
best/epoch,13
best/val_accuracy,0.27575
epoch,20


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 01 | Train Acc: 0.1422 | Val Acc: 0.1268 | Val Loss: 3.3251
Epoch 02 | Train Acc: 0.2378 | Val Acc: 0.1537 | Val Loss: 3.2078
Epoch 03 | Train Acc: 0.3390 | Val Acc: 0.1474 | Val Loss: 3.2619
Epoch 04 | Train Acc: 0.4289 | Val Acc: 0.1696 | Val Loss: 3.4678
Epoch 05 | Train Acc: 0.4995 | Val Acc: 0.1775 | Val Loss: 3.5712
Epoch 06 | Train Acc: 0.5805 | Val Acc: 0.1981 | Val Loss: 3.7439
Epoch 07 | Train Acc: 0.6483 | Val Acc: 0.2029 | Val Loss: 4.0214
Epoch 08 | Train Acc: 0.7203 | Val Acc: 0.1902 | Val Loss: 4.3669
Epoch 09 | Train Acc: 0.7721 | Val Acc: 0.2060 | Val Loss: 4.6162
Epoch 10 | Train Acc: 0.8131 | Val Acc: 0.1918 | Val Loss: 4.6723
Epoch 11 | Train Acc: 0.8352 | Val Acc: 0.1918 | Val Loss: 4.8687
Epoch 12 | Train Acc: 0.8578 | Val Acc: 0.1965 | Val Loss: 5.0392
Epoch 13 | Train Acc: 0.8992 | Val Acc: 0.2060 | Val Loss: 4.9644
Epoch 14 | Train Acc: 0.9058 | Val Acc: 0.1981 | Val Loss: 5.1596
Epoch 15 | Train Acc: 0.9223 | Val Acc: 0.1933 | Val Loss: 5.2291
Epoch 16 |

# B. Skeleton

In [89]:
import torch
import torch.nn as nn


class BiLSTMAttention(nn.Module):

    def __init__(
        self,
        input_size=153,
        hidden_size=128,
        num_layers=2,
        num_classes=40,
        dropout=0.3
    ):
        super().__init__()
        self.frame_projection = nn.Sequential(

    nn.Linear(num_joints * spatial_dim, 256),
    nn.ReLU(),

    nn.Dropout(dropout),

    nn.Linear(256, spatial_dim),
    nn.ReLU()
        )

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        # 256 -> attention score
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        # --------------------------------
        # (B, 64, 17, 6)
        # --------------------------------

        batch_size = x.size(0)

        # --------------------------------
        # Flatten joints + coordinates
        # --------------------------------

        x = x.reshape(
            batch_size,
            x.size(1),
            -1
        )

        # (B, 64, 102)

        # --------------------------------
        # BiLSTM
        # --------------------------------

        output, _ = self.lstm(x)

        # (B, 64, 256)

        # --------------------------------
        # Attention
        # --------------------------------

        scores = self.attention(output)

        # (B, 64, 1)

        weights = torch.softmax(
            scores,
            dim=1
        )

        # --------------------------------
        # Weighted temporal representation
        # --------------------------------

        context = torch.sum(
            output * weights,
            dim=1
        )

        # (B, 256)

        # --------------------------------
        # Classification
        # --------------------------------

        logits = self.classifier(context)

        return logits

In [90]:
class S6TemporalAttentionModel(nn.Module):

    def __init__(
        self,
        input_size=204,
        projection_size=128,
        hidden_size=128,
        num_layers=2,
        num_heads=4,
        num_classes=40,
        dropout=0.3
    ):
        super().__init__()

        # -----------------------------
        # Input projection
        # -----------------------------
        self.input_projection = nn.Sequential(
            nn.Linear(input_size, projection_size),
            nn.LayerNorm(projection_size),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # -----------------------------
        # BiLSTM
        # -----------------------------
        self.lstm = nn.LSTM(
            input_size=projection_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        feature_size = hidden_size * 2

        # -----------------------------
        # Multi-head temporal attention
        # -----------------------------
        self.temporal_attention = nn.MultiheadAttention(
            embed_dim=feature_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        # -----------------------------
        # Residual normalization
        # -----------------------------
        self.norm = nn.LayerNorm(feature_size)

        # -----------------------------
        # LEARNED TEMPORAL WEIGHTS
        # -----------------------------
        self.frame_attention = nn.Sequential(
            nn.Linear(feature_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        # -----------------------------
        # Classifier
        # -----------------------------
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        # B,T,J,F
        B, T, J, F = x.shape

        # -----------------------------
        # Flatten joints
        # -----------------------------
        x = x.reshape(B, T, J * F)

        # B,T,204
        # -----------------------------
        # Project
        # -----------------------------
        x = self.input_projection(x)

        # B,T,128

        # -----------------------------
        # BiLSTM
        # -----------------------------
        x, _ = self.lstm(x)

        # B,T,256

        # -----------------------------
        # Multi-head self attention
        # -----------------------------
        attended, _ = self.temporal_attention(
            x, x, x
        )

        x = self.norm(x + attended)

        # B,T,256

        # -----------------------------
        # LEARN FRAME IMPORTANCE
        # -----------------------------
        scores = self.frame_attention(x)

        # B,T,1

        weights = torch.softmax(
            scores,
            dim=1
        )

        # -----------------------------
        # Weighted temporal representation
        # -----------------------------
        x = torch.sum(
            x * weights,
            dim=1
        )

        # B,256

        # -----------------------------
        # Classification
        # -----------------------------
        logits = self.classifier(x)

        return logits

In [91]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = S6TemporalAttentionModel(
    input_size=204,
    projection_size=128,
    hidden_size=128,
    num_layers=2,
    num_heads=4,
    num_classes=40,
    dropout=0.3
).to(device)

print(model)
print("Device:", device)

S6TemporalAttentionModel(
  (input_projection): Sequential(
    (0): Linear(in_features=204, out_features=128, bias=True)
    (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
  )
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (temporal_attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
  )
  (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (frame_attention): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=1, bias=True)
  )
  (classifier): Sequential(
    (0): Dropout(p=0.3, inplace=False)
    (1): Linear(in_features=256, out_features=128, bias=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=40, bias=True)
  )


In [92]:
X, y = next(iter(train_loader))

X = X.to(device)

with torch.no_grad():
    output = model(X)

print("Input :", X.shape)
print("Output:", output.shape)
X, y = next(iter(train_loader))

print("Train batch X:", X.shape)
print("Train batch y:", y.shape)

X, y = next(iter(val_loader))

print("Val batch X:", X.shape)
print("Val batch y:", y.shape)

ValueError: not enough values to unpack (expected 4, got 3)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [ ]:
wandb.init(
    project="CIUX",
    name="Spatial BiLSTM S6",
    config={
        "model": "BiLSTM",
        "sequence_length": 64,
        "num_keypoints": 17,
        "coordinates": 3,
        "hidden_size": 128,
        "num_layers": 2,
        "dropout": 0.3,
        "batch_size": 32,
        "learning_rate": 1e-3,
        "epochs": 50,
        "optimizer": "Adam",
    }
)

In [ ]:
epochs = 20
best_val_accuracy = 0.0
for epoch in range(epochs):

    # ==========================================================
    # TRAIN
    # ==========================================================

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X, y in train_loader:

        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        output = model(X)

        loss = criterion(output, y)

        loss.backward()

        optimizer.step()

        train_loss += loss.item() * X.size(0)

        predictions = output.argmax(dim=1)

        train_correct += (predictions == y).sum().item()
        train_total += y.size(0)

    train_loss /= train_total
    train_accuracy = train_correct / train_total


    # ==========================================================
    # VALIDATION
    # ==========================================================

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for X, y in val_loader:

            X = X.to(device)
            y = y.to(device)

            output = model(X)

            loss = criterion(output, y)

            val_loss += loss.item() * X.size(0)

            predictions = output.argmax(dim=1)

            val_correct += (predictions == y).sum().item()
            val_total += y.size(0)

    val_loss /= val_total
    val_accuracy = val_correct / val_total
    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            "best_bilstm.pt"
        )

        print(
            f"🔥 New best model: "
            f"{best_val_accuracy:.4f}"
        )


    # ==========================================================
    # PRINT
    # ==========================================================

    print(
        f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )


    # ==========================================================
    # W&B
    # ==========================================================

    wandb.log({
        "epoch": epoch + 1,

        "train/loss": train_loss,
        "train/accuracy": train_accuracy,

        "val/loss": val_loss,
        "val/accuracy": val_accuracy,

        "learning_rate": optimizer.param_groups[0]["lr"]
    })

In [ ]:
wandb.finish()

---

# Z. Inference Part

## A. Test Dataset

In [ ]:
class SkeletonTestDataset(SkeletonDataset):

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        skeleton, confidence = self.load_skeleton(
            row["skeleton"]
        )

        # --------------------------------
        # No skeleton
        # --------------------------------
        if skeleton is None:

            skeleton = np.zeros(
                (
                    self.sequence_length,
                    17,
                    3
                ),
                dtype=np.float32
            )

        # --------------------------------
        # SAME preprocessing as training
        # --------------------------------

        skeleton = self.normalize_skeleton(
            skeleton
        )

        # Velocity
        velocity = np.diff(
            skeleton,
            axis=0,
            prepend=skeleton[0:1]
        )

        # Acceleration
        acceleration = np.diff(
            velocity,
            axis=0,
            prepend=velocity[0:1]
        )

        # Bone vectors
        bone_vectors = self.get_bone_vectors(
            skeleton
        )

        # --------------------------------
        # 12 features per joint
        # --------------------------------

        features = np.concatenate(
            [
                skeleton,
                velocity,
                acceleration,
                bone_vectors,
            ],
            axis=2
        )

        # --------------------------------
        # Temporal resampling
        # --------------------------------

        features = self.temporal_resample(
            features
        )

        # --------------------------------
        # Tensor
        # --------------------------------

        X = torch.tensor(
            features,
            dtype=torch.float32
        )

        return X, row["trial_id"]

## B. Defining the Path

In [ ]:
from pathlib import Path

BASE = Path("/kaggle/input/datasets/samasiayushman/small-model-track/Testing/Testing/small_model_track_test-007/small_model_track_test")

for p in BASE.rglob("SM_test_0001"):
    print("Found:", p)

In [ ]:
TEST_ROOT = p.parent
print(TEST_ROOT)

## 3. Testing Data

In [ ]:
TEST_ROOT = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "small-model-track/Testing/Testing/"
    "small_model_track_test-007/small_model_track_test"
)

test_records = []

for trial_dir in sorted(TEST_ROOT.iterdir()):

    if not trial_dir.is_dir():
        continue

    # Ignore .claude or any non-test directory
    if not trial_dir.name.startswith("SM_test_"):
        continue

    skeleton_path = trial_dir / "Skeleton"

    if not skeleton_path.is_dir():
        continue

    test_records.append({
        "trial_id": trial_dir.name,
        "path": str(trial_dir),
        "skeleton": str(skeleton_path)
    })

test_df = pd.DataFrame(test_records)

print(test_df.head())
print("Number of test trials:", len(test_df))

## 4. Calling the Dataset & Loader 

In [ ]:
test_dataset = SkeletonTestDataset(
    test_df,
    sequence_length=64
)

print("Test trials:", len(test_dataset))

In [ ]:
X, trial_id = test_dataset[0]

print("Trial:", trial_id)
print("Shape:", X.shape)
print("dtype:", X.dtype)
print("Min:", X.min().item())
print("Max:", X.max().item())

In [ ]:
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

X, trial_ids = next(iter(test_loader))

print("Batch X:", X.shape)
print("Trial IDs:", trial_ids[:5])

## 5. Evaluating

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/best_bilstm.pt"))
model.eval()

all_predictions = []
all_trial_ids = []

with torch.no_grad():

    for X, trial_ids in test_loader:

        X = X.to(device)

        logits = model(X)

        predictions = torch.argmax(logits, dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_trial_ids.extend(trial_ids)

## 6. Verifying Submission

In [ ]:
print("Number of predictions:", len(all_predictions))
print("Number of trial IDs:", len(all_trial_ids))

print("\nFirst predictions:")
for trial_id, pred in zip(
    all_trial_ids[:10],
    all_predictions[:10]
):
    print(trial_id, "->", pred)

In [ ]:
from collections import Counter

prediction_counts = Counter(all_predictions)

print("Predicted classes:")
for label, count in sorted(prediction_counts.items()):
    print(f"{label:2d}: {count}")

## 7. Submission

In [ ]:
submission = pd.DataFrame({
    "path": [
        f"small_model_track_test/{trial_id}/"
        for trial_id in all_trial_ids
    ],
    "prediction": all_predictions
})

print(submission.head())
print(submission.shape)

submission.to_csv("submission.csv",index=False)